# LLM Infrastructure: how thousands of GPUs train one model

Companion notebook to the second bonus lesson of **Transformers From Scratch**. Our models trained on
one machine. Llama 3 405B was trained on up to 16,384 H100 GPUs (Grattafiori et al., 2024), and
DeepSeek-V3 on a cluster of 2,048 H800 GPUs (DeepSeek-AI, 2024). Doing that takes a stack of
technologies that each solve one part of the problem:

| Layer | Technology | Responsibility |
|---|---|---|
| Compute | **CUDA** | programming model and platform for computation on NVIDIA GPUs |
| Inside a server | **NVLink** | high-speed GPU-to-GPU links |
| | **NVSwitch** | switch chips that connect every GPU in an NVLink domain to every other |
| Between servers | **RDMA** | moving data directly between the memory of two machines, without involving their CPUs |
| | **InfiniBand** | a high-performance network built around RDMA |
| | **RoCE** | RDMA over (lossless) Ethernet |
| | **GPUDirect RDMA** | lets the network card read and write GPU memory directly |
| Communication library | **NCCL** | collective operations (all-reduce, all-gather, ...) between GPUs, over whatever links exist |
| Training frameworks | **PyTorch DDP** | data parallelism: replicate the model, split the batch, average the gradients |
| | **Megatron-LM** | model parallelism: split the model itself (tensor and pipeline parallelism) |

The names are NVIDIA's, because NVIDIA's stack is the one these papers use. Other vendors provide
counterparts; AMD's RCCL, for example, provides "multi-GPU and multi-node collective communication
primitives optimized for AMD GPUs" (AMD ROCm documentation).

**What's inside.** Each section explains one layer of the stack and gives a small, runnable example
of the idea behind it: CUDA thread indexing, the collectives NCCL provides and the ring all-reduce, a
complete multi-process PyTorch DDP program, Megatron-style tensor parallelism, and the pipeline
schedule. Bandwidth figures come from vendor documentation and are used in idealized cost estimates.

**Sources.** Every factual claim is cited inline and listed under *References*.

In [1]:
import json, math, os, socket, subprocess, sys, tempfile, time
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.6.0


## 1. CUDA: programming the GPU

A GPU runs a function, called a **kernel**, on many threads at once. In NVIDIA's words: "When an
application launches a kernel, it does so with many threads, often millions of threads. These threads
are organized into blocks ... Thread blocks are organized into a grid." All threads of a block run on
one streaming multiprocessor (SM), where they can "communicate and synchronize with each other
efficiently" through on-chip **shared memory**. Within a block, threads are grouped into **warps** of
32 that execute in a Single-Instruction Multiple-Threads (SIMT) fashion (CUDA Programming Guide,
§1.2).

The guide's first example adds two vectors with one thread per element:

```cuda
__global__ void vecAdd(float* A, float* B, float* C, int vectorLength)
{
    // calculate which element this thread is responsible for computing
    int workIndex = threadIdx.x + blockDim.x * blockIdx.x;
    if (workIndex < vectorLength)
    {
        // Perform computation
        C[workIndex] = A[workIndex] + B[workIndex];
    }
}

// launch 4 thread blocks of 256 threads: one thread per element of a 1024-element vector
vecAdd<<<4, 256>>>(A, B, C, 1024);
```

`<<<grid, block>>>` is the *execution configuration*. Each thread computes its own index from its
block index and its position in the block; the bounds check lets the launch round the thread count up
to a whole number of blocks. The cell below reproduces this indexing in Python and checks that every
element is written exactly once.

In [2]:
def launch_vecadd(A, B, grid, block):
    # the indexing of vecAdd<<<grid, block>>>(A, B, C, len(A)), one thread at a time
    C = torch.full_like(A, float("nan"))
    writes = torch.zeros(len(A), dtype=torch.int64)
    idle = 0
    for blockIdx in range(grid):
        for threadIdx in range(block):
            workIndex = threadIdx + block * blockIdx       # threadIdx.x + blockDim.x * blockIdx.x
            if workIndex < len(A):                          # bounds check
                C[workIndex] = A[workIndex] + B[workIndex]
                writes[workIndex] += 1
            else:
                idle += 1
    return C, writes, idle

for N in (1024, 1000):
    A, B = torch.randn(N), torch.randn(N)
    block = 256
    grid = math.ceil(N / block)                             # round up to whole blocks
    C, writes, idle = launch_vecadd(A, B, grid, block)
    print(f"N = {N}: vecAdd<<<{grid}, {block}>>> -> {grid * block} threads ({block // 32} warps per block), "
          f"{idle} idle | correct: {torch.equal(C, A + B)} | every element written once: {bool((writes == 1).all())}")

N = 1024: vecAdd<<<4, 256>>> -> 1024 threads (8 warps per block), 0 idle | correct: True | every element written once: True
N = 1000: vecAdd<<<4, 256>>> -> 1024 threads (8 warps per block), 24 idle | correct: True | every element written once: True


## 2. NCCL: collective communication

"The NVIDIA Collective Communications Library (NCCL, pronounced 'Nickel') is a library providing
inter-GPU communication primitives that are topology-aware". It provides AllReduce, Broadcast, Reduce,
AllGather, ReduceScatter, AlltoAll, Gather and Scatter, plus point-to-point send and receive, and it
"supports a variety of interconnect technologies including PCIe, NVLINK, InfiniBand Verbs, and IP
sockets". Its API "closely follows the popular collectives API defined by MPI" (NCCL documentation).

The cell below defines what each collective computes, with each "GPU" (rank) represented by one
tensor, and checks two identities used throughout distributed training: an all-reduce equals a
reduce-scatter followed by an all-gather, and equals a reduce followed by a broadcast (the latter is
noted in the NCCL documentation).

In [3]:
def all_reduce(xs):     s = sum(xs); return [s.clone() for _ in xs]                 # everyone gets the sum
def reduce(xs, root=0): return sum(xs)                                             # only the root gets it
def broadcast(x, n):    return [x.clone() for _ in range(n)]                        # copy root's data to all
def reduce_scatter(xs): return list(sum(xs).chunk(len(xs)))                         # rank i gets chunk i of the sum
def all_gather(xs):     full = torch.cat(xs); return [full.clone() for _ in xs]     # everyone gets everything
def all_to_all(xs):                                                                 # rank j gets chunk j from every rank
    parts = [x.chunk(len(xs)) for x in xs]
    return [torch.cat([parts[src][dst] for src in range(len(xs))]) for dst in range(len(xs))]

n = 4
xs = [torch.arange(8.0) + 100 * r for r in range(n)]          # 4 ranks, 8 numbers each
same = lambda a, b: all(torch.equal(u, v) for u, v in zip(a, b))
print("all-reduce == reduce-scatter + all-gather:", same(all_reduce(xs), all_gather(reduce_scatter(xs))))
print("all-reduce == reduce + broadcast:         ", same(all_reduce(xs), broadcast(reduce(xs), n)))
print("rank 1 after all-to-all:", all_to_all(xs)[1].tolist())

all-reduce == reduce-scatter + all-gather: True
all-reduce == reduce + broadcast:          True
rank 1 after all-to-all: [2.0, 3.0, 102.0, 103.0, 202.0, 203.0, 302.0, 303.0]


### Ring all-reduce

How is an all-reduce actually performed? The NCCL tests documentation analyses it: for each element
an all-reduce needs $n-1$ additions and $n$ assignments, so "we need 2(n-1) data transfers", and if
each of the $n$ ranks has bandwidth $B$, an all-reduce of $S$ bytes takes at best

$$t = \frac{S}{B} \cdot \frac{2(n-1)}{n} ,$$

"independent of the algorithm used (ring, tree, or other)". A **ring** reaches this bound. The data is
cut into $n$ chunks. In a *reduce-scatter* phase of $n-1$ steps, every rank passes one chunk to its
neighbour, which adds it to its own copy; afterwards each rank holds the complete sum of one chunk. In
an *all-gather* phase of another $n-1$ steps, the completed chunks travel around the ring. Each rank
sends $2(n-1)$ chunks of size $S/n$: about $2S$, however many ranks there are.

In [4]:
def ring_all_reduce(xs):
    n = len(xs)
    chunks = [list(x.clone().chunk(n)) for x in xs]          # chunks[r][c]: rank r's copy of chunk c
    sent = [0] * n                                            # elements sent by each rank
    # phase 1, reduce-scatter: in step k, rank r sends chunk (r - k) % n to rank r + 1, which adds it
    for k in range(n - 1):
        msgs = [(r, (r - k) % n, chunks[r][(r - k) % n].clone()) for r in range(n)]   # all send at once
        for r, c, data in msgs:
            chunks[(r + 1) % n][c] += data
            sent[r] += data.numel()
    # now rank r holds the complete sum of chunk (r + 1) % n
    # phase 2, all-gather: in step k, rank r forwards chunk (r + 1 - k) % n, which rank r + 1 copies
    for k in range(n - 1):
        msgs = [(r, (r + 1 - k) % n, chunks[r][(r + 1 - k) % n].clone()) for r in range(n)]
        for r, c, data in msgs:
            chunks[(r + 1) % n][c] = data
            sent[r] += data.numel()
    return [torch.cat(c) for c in chunks], sent

S = 1024                                                      # elements per rank
print(f"{'ranks':>5s} | {'correct':>7s} | {'sent per rank (ring)':>20s} | {'theory 2(n-1)/n':>15s} | "
      f"{'received by rank 0, gather-to-root':>34s}")
for n in (2, 4, 8, 16):
    xs = [torch.randn(S, dtype=torch.float64) for _ in range(n)]
    out, sent = ring_all_reduce(xs)
    ok = all(torch.allclose(o, sum(xs)) for o in out)
    print(f"{n:5d} | {str(ok):>7s} | {max(sent) / S:18.3f} S | {2 * (n - 1) / n:13.3f} S | {n - 1:32d} S")
for n in (1024,):
    print(f"{n:5d} | {'':>7s} | {'':>20s} | {2 * (n - 1) / n:13.3f} S | {n - 1:32d} S")

ranks | correct | sent per rank (ring) | theory 2(n-1)/n | received by rank 0, gather-to-root
    2 |    True |              1.000 S |         1.000 S |                                1 S
    4 |    True |              1.500 S |         1.500 S |                                3 S
    8 |    True |              1.750 S |         1.750 S |                                7 S
   16 |    True |              1.875 S |         1.875 S |                               15 S
 1024 |         |                      |         1.998 S |                             1023 S


**Reading the output.** The ring produces the exact sum on every rank, and the measured traffic per
rank matches $2(n-1)/n \cdot S$, approaching $2S$ as $n$ grows. Sending everything to one root
instead makes that rank receive $(n-1)S$, which grows linearly with the number of GPUs. (NCCL also
implements tree algorithms, which reduce latency at large scale; the bandwidth bound above applies to
all of them.)

## 3. NVLink, NVSwitch, PCIe and the network: why bandwidth decides the design

**Inside a server.** H100 GPUs have 18 fourth-generation NVLink links, each delivering "25 GB/sec
effective bandwidth in each direction", for "900 GB/sec total bandwidth", which is "7x the bandwidth of
PCIe Gen 5"; PCIe Gen 5 provides "128 GB/sec total bandwidth (64 GB/sec in each direction)" (NVIDIA
Hopper architecture blog). NVIDIA's NVLink specifications list 900 GB/s per GPU for the Hopper
generation, 1,800 GB/s for Blackwell and 3,000 GB/s for Vera Rubin (preliminary). **NVSwitch** chips
connect all GPUs of an NVLink domain to one another: in a DGX H100 server, eight H100 GPUs, with
"900 GB/s GPU-to-GPU bandwidth" (DGX H100 user guide). NVLink domains are no longer limited to one
server: rack-scale systems such as Blackwell NVL72 connect 72 GPUs with NVLink Switch chips (NVIDIA
NVLink specifications).

**Between servers.** The same DGX H100 has eight single-port ConnectX-7 network cards, one per GPU,
each "up to 400Gbps" InfiniBand, or Ethernet (DGX H100 user guide). 400 Gb/s is 50 GB/s in each
direction: one ninth of NVLink.

The cell below turns these peak numbers into **idealized lower bounds** for an all-reduce across
8 GPUs, using the formula above: no latency, no protocol overhead, perfect use of the link. Achieved
bandwidth is lower, which is why the NCCL tests report a separate "bus bandwidth".

In [5]:
links = {                                         # peak bandwidth per GPU, each direction, bytes/s
    "NVLink 4 (H100)":           18 * 25e9,       # 18 links x 25 GB/s
    "PCIe Gen5 x16":             64e9,
    "InfiniBand 400 Gb/s NIC":   400e9 / 8,
}

def allreduce_seconds(nbytes, n, bw):             # ring bound: (S / B) * 2(n - 1) / n
    return nbytes / bw * 2 * (n - 1) / n

n = 8
grads = 8e9 * 2                                   # data parallelism: gradients of an 8B model in bf16
act = 1 * 8192 * 16384 * 2                        # tensor parallelism: one activation all-reduce for a
                                                  # Llama 3 405B layer (1 sequence x 8192 tokens x
                                                  # hidden size 16,384, bf16)
print(f"all-reduce across {n} GPUs   | {grads / 1e9:.0f} GB of gradients (once per step) | "
      f"{act / 2**20:.0f} MiB of activations (4 per layer, per micro-batch)")
for name, bw in links.items():
    print(f"{name:25s} | {allreduce_seconds(grads, n, bw):9.3f} s{'':27s} | "
          f"{1e3 * allreduce_seconds(act, n, bw):6.2f} ms")

all-reduce across 8 GPUs   | 16 GB of gradients (once per step) | 256 MiB of activations (4 per layer, per micro-batch)
NVLink 4 (H100)           |     0.062 s                            |   1.04 ms
PCIe Gen5 x16             |     0.438 s                            |   7.34 ms
InfiniBand 400 Gb/s NIC   |     0.560 s                            |   9.40 ms


**Reading the output.** A data-parallel gradient all-reduce happens once per training step and can
overlap with the backward pass (section 5); even at network speed it takes about half a second in
this idealized estimate. Tensor parallelism all-reduces activations four times per transformer layer
(section 6), for every micro-batch, and the layer cannot continue until each all-reduce finishes: at
network speed that is about 9 ms per all-reduce instead of 1 ms over NVLink. That is why Megatron-LM's authors found "tensor model parallelism
is effective within a multi-GPU server", where NVLink is available, "but pipeline model parallelism
must be used for larger models" (Narayanan et al., 2021).

## 4. RDMA, InfiniBand, RoCE and GPUDirect RDMA

How data moves between servers:

- **RDMA.** "Remote Direct Memory Access (RDMA) is the remote memory management capability that
  allows server-to-server data movement directly between application memory without any CPU
  involvement" (NVIDIA networking documentation, *RDMA over Converged Ethernet*).
- **InfiniBand** is a network built for this. NVIDIA's Quantum-2 InfiniBand platform offers RDMA,
  "In-Network Computing" engines such as the Scalable Hierarchical Aggregation and Reduction Protocol
  (SHARPv3), and speeds "up to 400Gb/s" (NVIDIA Quantum-2 product page).
- **RoCE**, RDMA over Converged Ethernet, provides "this efficient data transfer with very low
  latencies on lossless Ethernet networks". RoCEv1 uses a dedicated Ethernet type (0x8915); RoCEv2
  runs over UDP on port 4791, so it can be routed over IP. "RoCE requires a form of flow control", in
  practice Priority Flow Control (PFC) "enabled on all endpoints and switches in the flow path"
  (NVIDIA networking documentation).
- **GPUDirect RDMA** "enables a direct path for data exchange between the GPU and a third-party peer
  device using standard features of PCI Express", such as a network card; it was introduced with
  Kepler-class GPUs and CUDA 5.0, and typically requires the two devices to "share the same upstream
  PCI Express root complex" (NVIDIA GPUDirect RDMA documentation). It "eliminates the system CPUs and
  the required buffer copies of data via the system memory" (NVIDIA GPUDirect overview). Without it,
  data leaving a GPU for the network is first copied to a buffer in host memory.
- **PyTorch** recommends the NCCL backend on GPU hosts with InfiniBand, "since it's the only backend
  that currently supports InfiniBand and GPUDirect" (PyTorch distributed documentation).

**A real example.** "Llama 3 405B used RDMA over Converged Ethernet (RoCE) fabric ... Smaller models
in the Llama 3 family were trained using Nvidia Quantum2 Infiniband fabric. Both RoCE and Infiniband
clusters leverage 400 Gbps interconnects between GPUs", and "within a server, the eight GPUs are
connected via NVLink" (Grattafiori et al., 2024).

## 5. PyTorch DDP: data parallelism

Distributed data parallelism "replicates the model on every computational resource to generate
gradients independently and then communicates those gradients at each iteration to keep model
replicas consistent" (Li et al., 2020). In PyTorch:

- One process runs per GPU. At construction, DDP "broadcasts state_dict() from the process with rank 0
  to all other processes in the group to make sure that all model replicas start from the exact same
  state".
- Gradients are grouped into **buckets**. Autograd hooks fire as each gradient becomes ready during
  the backward pass, and a bucket is all-reduced as soon as it is full, so communication overlaps with
  the rest of the backward pass (PyTorch DDP design note). Li et al. report "near-linear scalability
  using 256 GPUs".
- PyTorch recommends the NCCL backend for distributed training on CUDA GPUs and Gloo on CPUs
  (PyTorch distributed documentation).

**Experiment.** The script below is a complete DDP program. `torchrun` starts 4 processes, one per
GPU, using the NCCL backend on GPUs (Gloo otherwise). Each process trains on its own quarter of the batch,
and each starts from a *different* random initialisation on purpose. If DDP works as described,
(1) all four processes end with identical weights, and (2) those weights equal a single-process run
on the full batch that starts from rank 0's initialisation, because averaging the four shards'
gradients gives the full-batch gradient. The experiment is repeated for 3 random datasets.

In [6]:
DDP_SCRIPT = r"""
import json, os, sys
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(16, 64), nn.Tanh(), nn.Linear(64, 1))

def make_data(trial):
    g = torch.Generator().manual_seed(100 + trial)
    X = torch.randn(256, 16, generator=g)
    return X, torch.sin(X.sum(1, keepdim=True))

def main():
    gpu = torch.cuda.is_available()
    dist.init_process_group("nccl" if gpu else "gloo")
    rank, world = dist.get_rank(), dist.get_world_size()
    dev = torch.device(f"cuda:{os.environ['LOCAL_RANK']}" if gpu else "cpu")
    results = {}
    for trial in range(3):
        X, y = (t.to(dev) for t in make_data(trial))
        n = len(X) // world
        lo, hi = rank * n, (rank + 1) * n     # this rank's shard
        model = DDP(make_model(seed=rank).to(dev))   # different init per rank
        opt = torch.optim.SGD(model.parameters(), lr=0.1)
        for step in range(50):
            loss = F.mse_loss(model(X[lo:hi]), y[lo:hi])
            opt.zero_grad()
            loss.backward()                   # DDP averages gradients here
            opt.step()
        weights = [p.detach().flatten() for p in model.parameters()]
        results[trial] = torch.cat(weights).tolist()
    path = os.path.join(sys.argv[1], f"rank{rank}.json")
    with open(path, "w") as f:
        json.dump(results, f)
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
"""

In [7]:
workdir = Path(tempfile.mkdtemp(prefix="ddp_demo_"))
script = workdir / "ddp_demo.py"
script.write_text(DDP_SCRIPT)

with socket.socket() as s:                                   # pick a free local port
    s.bind(("127.0.0.1", 0)); port = s.getsockname()[1]
world = 4
t0 = time.time()
run = subprocess.run([sys.executable, "-m", "torch.distributed.run", f"--nproc_per_node={world}",
                      "--master_addr=127.0.0.1", f"--master_port={port}", str(script), str(workdir)],
                     capture_output=True, text=True, timeout=600)
assert run.returncode == 0, run.stderr[-2000:]
print(f"torchrun: {world} processes finished in {time.time() - t0:.1f} s")

# single-process reference, using the script's own model and data functions
import importlib.util
spec = importlib.util.spec_from_file_location("ddp_demo", script)
demo = importlib.util.module_from_spec(spec); spec.loader.exec_module(demo)
ranks = [json.loads((workdir / f"rank{r}.json").read_text()) for r in range(world)]
for trial in range(3):
    X, y = demo.make_data(trial)
    ref = demo.make_model(seed=0)                             # rank 0's initialisation
    opt = torch.optim.SGD(ref.parameters(), lr=0.1)
    for step in range(50):
        loss = F.mse_loss(ref(X), y)                          # the whole batch at once
        opt.zero_grad(); loss.backward(); opt.step()
    ref_w = torch.cat([p.detach().flatten() for p in ref.parameters()])
    w = [torch.tensor(r[str(trial)]) for r in ranks]
    spread = max((wi - w[0]).abs().max().item() for wi in w)
    print(f"dataset {trial}: max difference between ranks {spread:.1e} | "
          f"rank 0 vs single process {(w[0] - ref_w).abs().max().item():.1e} | "
          f"rank 1's own init vs rank 0's: {(demo.make_model(1)[0].weight - demo.make_model(0)[0].weight).abs().max().item():.2f}")

torchrun: 4 processes finished in 1.9 s


dataset 0: max difference between ranks 0.0e+00 | rank 0 vs single process 3.0e-08 | rank 1's own init vs rank 0's: 0.49
dataset 1: max difference between ranks 0.0e+00 | rank 0 vs single process 6.0e-08 | rank 1's own init vs rank 0's: 0.49
dataset 2: max difference between ranks 0.0e+00 | rank 0 vs single process 3.0e-08 | rank 1's own init vs rank 0's: 0.49


**Reading the output.** The four processes end with identical weights even though each started from
its own random initialisation (the last column shows how different those were): DDP's initial
broadcast made them start equal, and the averaged gradients kept them equal. Their weights match the
single-process full-batch run up to floating-point rounding, because the all-reduce adds the
gradients in a different order than a single process does.

## 6. Megatron-LM: tensor parallelism

When the model does not fit on one GPU, its layers must be split. Shoeybi et al. (2019) split the
transformer's MLP, $Z = \mathrm{GeLU}(XA)\,B$, across GPUs. Splitting $A$ by rows would require
summing partial products *before* the GeLU, "since GeLU is a nonlinear function". Instead they "split
A along its columns", so each GPU applies the GeLU to its own slice, and "split the second GEMM along
its rows so it takes the output of the GeLU layer directly without requiring any communication". The
partial outputs are then summed across GPUs with one all-reduce. Attention is split the same way, by
heads. The result: "all GEMMs in a simple transformer layer using only two all-reduces in the forward
path and two in the backward path".

The cell below simulates 4 GPUs in one process and compares both splits with the unsplit layer.

In [8]:
torch.manual_seed(0)
h, ffn, t = 64, 256, 4                          # hidden size, MLP size, tensor-parallel degree ("GPUs")
X = torch.randn(10, h, dtype=torch.float64)     # 10 tokens
A = torch.randn(h, ffn, dtype=torch.float64) / h**0.5
B = torch.randn(ffn, h, dtype=torch.float64) / ffn**0.5
dense = F.gelu(X @ A) @ B

# Megatron: GPU i holds a column block of A and the matching row block of B
A_cols, B_rows = A.chunk(t, dim=1), B.chunk(t, dim=0)
partial = [F.gelu(X @ A_cols[i]) @ B_rows[i] for i in range(t)]    # no communication needed
megatron = sum(partial)                                            # the forward pass's one all-reduce

# the alternative: split A by rows (and X by columns); the GeLU then sees partial sums
A_rows, X_cols = A.chunk(t, dim=0), X.chunk(t, dim=1)
row_split = sum(F.gelu(X_cols[i] @ A_rows[i]) for i in range(t)) @ B

print(f"MLP | column-then-row split vs unsplit: max difference {(megatron - dense).abs().max():.1e}")
print(f"MLP | row split of A vs unsplit:        max difference {(row_split - dense).abs().max():.1e}")

def attention(X, Wq, Wk, Wv, Wo, heads):
    T = X.shape[0]
    q, k, v = ((X @ W).view(T, heads, -1).transpose(0, 1) for W in (Wq, Wk, Wv))
    att = torch.softmax(q @ k.transpose(1, 2) / math.sqrt(q.shape[-1]), dim=-1) @ v
    return att.transpose(0, 1).reshape(T, -1) @ Wo

n_heads = 8
Wq, Wk, Wv, Wo = (torch.randn(h, h, dtype=torch.float64) / h**0.5 for _ in range(4))
dense_att = attention(X, Wq, Wk, Wv, Wo, n_heads)
cols = lambda W: W.chunk(t, dim=1)                                # GPU i: heads 2i and 2i+1
tp_att = sum(attention(X, cols(Wq)[i], cols(Wk)[i], cols(Wv)[i], Wo.chunk(t, dim=0)[i], n_heads // t)
             for i in range(t))                                   # the second all-reduce
print(f"attention | heads split over {t} GPUs vs unsplit: max difference {(tp_att - dense_att).abs().max():.1e}")
print(f"weights per GPU: 1/{t} of the layer | all-reduces per layer: 2 forward + 2 backward")

MLP | column-then-row split vs unsplit: max difference 1.8e-15
MLP | row split of A vs unsplit:        max difference 1.1e+00
attention | heads split over 4 GPUs vs unsplit: max difference 6.7e-16
weights per GPU: 1/4 of the layer | all-reduces per layer: 2 forward + 2 backward


## 7. Pipeline parallelism and the bubble

Pipeline parallelism gives each GPU a consecutive group of layers (a *stage*) and splits each batch
into $m$ micro-batches that flow through the $p$ stages (Huang et al., 2019). At the start and end of
every batch some stages have nothing to do. For the schedule that runs all forward passes and then all
backward passes, Narayanan et al. (2021) derive the bubble time
$t_{pb} = (p-1)(t_f + t_b)$, so the bubble is $(p-1)/m$ of the ideal computation time
$m(t_f + t_b)$: "for the bubble time fraction to be small, we thus need $m \gg p$". Their interleaved
schedule gives each GPU $v$ smaller stages and reduces the bubble to $\frac{1}{v} \cdot \frac{p-1}{m}$,
at the cost of $v$ times more communication.

The cell below simulates the schedule, with a backward pass taking twice as long as a forward pass,
and compares the measured bubble with the formula.

In [9]:
def pipeline_schedule(p, m, tf=1, tb=2):
    # all-forward, all-backward schedule; returns {stage: [(start, end, label)]}
    free, done_f, done_b = [0] * p, {}, {}
    ops = {s: [] for s in range(p)}
    for j in range(m):                                  # forward passes flow down the stages
        for s in range(p):
            start = max(free[s], done_f.get((s - 1, j), 0))
            done_f[(s, j)] = free[s] = start + tf
            ops[s].append((start, start + tf, "F"))
    for j in range(m):                                  # backward passes flow back up
        for s in reversed(range(p)):
            start = max(free[s], done_b.get((s + 1, j), 0))
            done_b[(s, j)] = free[s] = start + tb
            ops[s].append((start, start + tb, "B"))
    return ops

def bubble_fraction(ops, m, tf=1, tb=2):
    makespan = max(end for stage in ops.values() for _, end, _ in stage)
    return (makespan - m * (tf + tb)) / (m * (tf + tb))

ops = pipeline_schedule(p=4, m=8)
print("p = 4 stages, m = 8 micro-batches (F = forward, B = backward, . = idle):")
for s, stage in ops.items():
    row = ["."] * max(end for st in ops.values() for _, end, _ in st)
    for start, end, kind in stage:
        row[start:end] = kind * (end - start)
    print(f"  stage {s}: {''.join(row)}")
print()
for p, m in [(4, 4), (4, 8), (4, 32), (16, 16), (16, 64), (16, 256)]:
    print(f"p = {p:2d}, m = {m:3d}: simulated bubble {bubble_fraction(pipeline_schedule(p, m), m):.3f} "
          f"| formula (p - 1) / m = {(p - 1) / m:.3f}")

p = 4 stages, m = 8 micro-batches (F = forward, B = backward, . = idle):
  stage 0: FFFFFFFF.........BBBBBBBBBBBBBBBB
  stage 1: .FFFFFFFF......BBBBBBBBBBBBBBBB..
  stage 2: ..FFFFFFFF...BBBBBBBBBBBBBBBB....
  stage 3: ...FFFFFFFFBBBBBBBBBBBBBBBB......

p =  4, m =   4: simulated bubble 0.750 | formula (p - 1) / m = 0.750
p =  4, m =   8: simulated bubble 0.375 | formula (p - 1) / m = 0.375
p =  4, m =  32: simulated bubble 0.094 | formula (p - 1) / m = 0.094
p = 16, m =  16: simulated bubble 0.938 | formula (p - 1) / m = 0.938
p = 16, m =  64: simulated bubble 0.234 | formula (p - 1) / m = 0.234
p = 16, m = 256: simulated bubble 0.059 | formula (p - 1) / m = 0.059


## 8. Putting it together

Large runs combine all of these. Llama 3 405B used "4D parallelism": tensor (TP), context (CP),
pipeline (PP) and data parallelism (DP, in the form of FSDP). The groups are ordered [TP, CP, PP, DP]
because "the innermost parallelism requires the highest network bandwidth and lowest latency, and
hence is usually constrained to within the same server" (Grattafiori et al., 2024, Table 4 and
Figure 5). The reported configurations reached 38-43% BF16 model FLOPs utilization.

DeepSeek-V3 made a different choice: its authors optimized memory use enough to train "without using
costly tensor parallelism". Its experts communicate across nodes over InfiniBand, and because "NVLink
offers a bandwidth of 160 GB/s, roughly 3.2 times that of IB (50 GB/s)" on their H800 cluster, they
"limit each token to be dispatched to at most 4 nodes, thereby reducing IB traffic" (DeepSeek-AI,
2024).

In [10]:
gpus_per_server = 8                                         # Grand Teton servers: eight GPUs each
configs = [                                                 # Grattafiori et al. (2024), Table 4
    ("8K-token sequences",   dict(TP=8, CP=1,  PP=16, DP=64),  "43%"),
    ("8K-token sequences",   dict(TP=8, CP=1,  PP=16, DP=128), "41%"),
    ("128K-token sequences", dict(TP=8, CP=16, PP=16, DP=8),   "38%"),
]
for name, c, mfu in configs:
    total = math.prod(c.values())
    print(f"{name:20s}: TP {c['TP']} x CP {c['CP']:2d} x PP {c['PP']} x DP {c['DP']:3d} = {total:6,} GPUs "
          f"in {total // gpus_per_server:5,} servers | MFU {mfu} | TP group = one server: "
          f"{c['TP'] == gpus_per_server}")

8K-token sequences  : TP 8 x CP  1 x PP 16 x DP  64 =  8,192 GPUs in 1,024 servers | MFU 43% | TP group = one server: True
8K-token sequences  : TP 8 x CP  1 x PP 16 x DP 128 = 16,384 GPUs in 2,048 servers | MFU 41% | TP group = one server: True
128K-token sequences: TP 8 x CP 16 x PP 16 x DP   8 = 16,384 GPUs in 2,048 servers | MFU 38% | TP group = one server: True


## Scope and limitations

- The examples are deliberately small. They verify what each technique computes (exact sums,
  identical replicas, an unchanged layer output), not its speed at scale.
- Bandwidth figures are vendor peak specifications; the time estimates derived from them are lower
  bounds.

## References

**Systems and case studies**
- Grattafiori et al. (2024). *The Llama 3 Herd of Models.* [arXiv:2407.21783](https://arxiv.org/abs/2407.21783)
- DeepSeek-AI (2024). *DeepSeek-V3 Technical Report.* [arXiv:2412.19437](https://arxiv.org/abs/2412.19437)

**Parallel training**
- Li et al. (2020). *PyTorch Distributed: Experiences on Accelerating Data Parallel Training.* VLDB 2020. [arXiv:2006.15704](https://arxiv.org/abs/2006.15704)
- Shoeybi et al. (2019). *Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism.* [arXiv:1909.08053](https://arxiv.org/abs/1909.08053)
- Narayanan et al. (2021). *Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM.* SC 2021. [arXiv:2104.04473](https://arxiv.org/abs/2104.04473)
- Huang et al. (2019). *GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism.* [arXiv:1811.06965](https://arxiv.org/abs/1811.06965)

**Documentation**
- NVIDIA. *CUDA Programming Guide*, §1.2 Programming Model and §2.1 Intro to CUDA C++. [docs.nvidia.com/cuda/cuda-programming-guide](https://docs.nvidia.com/cuda/cuda-programming-guide/index.html)
- NVIDIA. *NCCL User Guide: Overview of NCCL* and *Collective Operations.* [docs.nvidia.com/deeplearning/nccl](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/overview.html)
- NVIDIA. *NCCL tests: Performance reported by NCCL tests.* [github.com/NVIDIA/nccl-tests](https://github.com/NVIDIA/nccl-tests/blob/master/doc/PERFORMANCE.md)
- NVIDIA. *NVIDIA Hopper Architecture In-Depth* (technical blog). [developer.nvidia.com](https://developer.nvidia.com/blog/nvidia-hopper-architecture-in-depth/)
- NVIDIA. *NVLink and NVLink Switch* (specifications). [nvidia.com](https://www.nvidia.com/en-us/data-center/nvlink/)
- NVIDIA. *DGX H100/H200 User Guide: Introduction.* [docs.nvidia.com/dgx](https://docs.nvidia.com/dgx/dgxh100-user-guide/introduction-to-dgxh100.html)
- NVIDIA. *Quantum-2 InfiniBand platform.* [nvidia.com](https://www.nvidia.com/en-us/networking/quantum2/)
- NVIDIA. *RDMA over Converged Ethernet (RoCE)*, MLNX_OFED documentation. [docs.nvidia.com/networking](https://docs.nvidia.com/networking/display/mlnxofedv24010331/rdma+over+converged+ethernet+(roce))
- NVIDIA. *Developing a Linux Kernel Module using GPUDirect RDMA.* [docs.nvidia.com/cuda/gpudirect-rdma](https://docs.nvidia.com/cuda/gpudirect-rdma/index.html)
- NVIDIA. *GPUDirect* (overview). [developer.nvidia.com/gpudirect](https://developer.nvidia.com/gpudirect)
- PyTorch. *Distributed communication package: Which backend to use?* [docs.pytorch.org](https://docs.pytorch.org/docs/stable/distributed.html)
- PyTorch. *Distributed Data Parallel* (design note). [docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/ddp.html)
- AMD. *RCCL documentation.* [rocm.docs.amd.com](https://rocm.docs.amd.com/projects/rccl/en/latest/)